In [1]:
import argparse
from typing import cast
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader, random_split
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import torch.nn as nn
from tqdm.notebook import tqdm
import zipfile
import os
import numpy as np
import math
from torchvision.transforms import v2
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Config
import transformers
import random
import mingpt.gen as gen
from mingpt.gen import unnormalize, device, train
import mingpt.data as data
import shlex
import argparse

import vqvae
import vqvae.utils
import json

In [2]:
"""

@dataclass
class ExperimentConfig():
    work_dir : str
    image_width : int
    image_height : int
    image_channel : int 
    image_token_length : int # = image_width * image_height * image_channel for pixel images, otherwise it depends on latent space size

    total_length : int # = image_token_length + text_codebook_size + special_token_num
    special_token_num : int # equals to 2, for <bos> and <eos>
    bos_id : int # 256 for pixel image
    eos_id : int # 257 for pixel image

    dataset_type : str # mnist or fashion mnist or...

    # tunable hyperparameter for training
    batch_size : int
    learning_rate : float
    n_epoch : int
    n_warmup : int = 1000

    # in steps. If you have a powerful gpu, you can upscale the value
    log_iter : int = 300
    save_iter : int = 1000
    eval_iter : int = 300

    # hyperparameter for model (default is 3M)
    n_embed : int = 256
    n_head : int = 8

    use_vqvae : bool = False # whether use vqvae to compress the image first

    image_codebook_size : int = 256 # for image this is [0,255] pixel value

    text_codebook_size : int = 0 # unconditional generation
"""
# Create the parser
parser = argparse.ArgumentParser()

timestamp = vqvae.utils.readable_timestamp()
# Add arguments

parser.add_argument('--dataset_type', type=str, help='type of dataset, can only be "mnist", "fashion", "spirit" or "face"')
# work directory of the model
# the plot and model will be saved to this directory
# I recommend you to use different work directory for different hyper parameters
parser.add_argument('--work_dir', type=str)
parser.add_argument('--data_dir', type=str)
# logging interval, you can increase this value if there are too many logs
parser.add_argument('--log_iter', type=int, default=300)
# checkpoint interval
parser.add_argument('--save_iter', type=int, default=1000)
# evaluation interval
parser.add_argument('--eval_iter', type=int, default=300)
# embedding size of GPT2 model
parser.add_argument('--n_embed', type=int, default=256)
# number of attention head
parser.add_argument('--n_head', type=int, default=8)
# number of layer
parser.add_argument('--n_layer', type=int)

# hyper parameter
parser.add_argument('--batch_size', type=int)
parser.add_argument('--learning_rate', type=float)
parser.add_argument('--n_epoch', type=int)
parser.add_argument('--pdropout', type=float)

# I prefer 1000 for learning rate, it should be around 1% of the total training steps
parser.add_argument('--n_warmup', type=int)
# set to 2 for mnist and fashion dataset, if you want a quick experiment
# this will downscale the image to a smaller size
parser.add_argument('--downscale', type=int)

# parameter used only in evaluation
# if you want to perform evaluation with a model, then provide the fullpath to a model.pt file
parser.add_argument('--checkpoint', type=str)
# row = 5 will generate 5*5 images
parser.add_argument('--row', type=int, default=5)
# higher temperature will increase diversity of output, you can tune this parameter by yourself
parser.add_argument('--temperature', type=float, default=1.0)
# try 2 to 5, will slow down generation
parser.add_argument('--num_beams', type=int, default=2)

# vqvae parameter
parser.add_argument('--vqvae_checkpoint', type=str)

# important!!!
# you must set seed to 0 during training for reproducibility 
# for evaluation you can ignore this parameter, then the script will pick a seed for you
parser.add_argument('--seed', type=int, default=random.randrange(100000))

# whether we do unconditional generation
parser.add_argument('--unconditional', action='store_true')
example = """
--dataset_type vqvae-flower \
       --work_dir work/mingpt \
       --log_iter 300 \
       --save_iter 1000 \
       --eval_iter 3000 \
       --n_embed 256 \
       --n_head 8 \
       --n_layer 4 \
       --pdropout 0.01 \
       --batch_size 16 \
       --learning_rate 6e-4 \
       --n_epoch 256 \
       --n_warmup 3000 \
       --downscale 1 \
       --row 5 \
       --num_beams 2 \
       --temperature 1 \
       --data_dir=data \
       --vqvae_checkpoint=work/vqvae/mon_apr_15_11_00_48_2024/results/flower-vqvae.pt \
"""
def in_ipython():
    try:
        return __IPYTHON__
    except NameError:
        return False
    
if in_ipython():
    args = parser.parse_args(shlex.split(example))
else:
    args = parser.parse_args()
print(args)
assert not args.unconditional

Namespace(dataset_type='vqvae-flower', work_dir='work/mingpt', data_dir='data', log_iter=300, save_iter=1000, eval_iter=3000, n_embed=256, n_head=8, n_layer=4, batch_size=16, learning_rate=0.0006, n_epoch=256, pdropout=0.01, n_warmup=3000, downscale=1, checkpoint=None, row=5, temperature=1.0, num_beams=2, vqvae_checkpoint='work/vqvae/mon_apr_15_11_00_48_2024/results/flower-vqvae.pt', seed=43534, unconditional=False)


In [3]:
from torchvision.transforms import v2
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Config
import transformers
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision.transforms as transforms
from mingpt.data import crop_and_resize_to_square

In [4]:
data_dir = "data"
def list_file(dir):
    return list(os.walk(dir))[0][-1]
import re
root_dir = "flower/text_c10"
dd = {}
for d in list(os.walk(os.path.join(data_dir, root_dir)))[0][1]:
    class_id = int(re.findall(r'\d+', d)[-1])
    d = os.path.join(data_dir, root_dir, d)
    for f in list_file(d):
        if f.endswith('txt'):
            image_id = re.findall(r'\d+', f)[-1]
            with open(os.path.join( d, f)) as ff:
                dd[image_id] = ff.read().split('\n')
all_texts = []
for i in dd:
    all_texts.extend(dd[i])

In [5]:
# from transformers import BertModel, BertPreTrainedModel, AutoTokenizer
# class TextEncoder(torch.nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
#         self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
#         self.text_encoder.eval()
#         self.embedding_size = 768
#     def encode(self, text):
#         with torch.no_grad():
#             self.eval()
#             if isinstance(text, str):
#                 text = text.lower()
#             else:
#                 text = [i.lower() for i in text]
#             token = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(self.text_encoder.device)
#             return self.text_encoder(**token).last_hidden_state[:,0,:]

In [6]:
from vqvae.model import VQVAE, CodeLayer
from vqvae.unet_model import UNetVQVAE
import vqvae
vqvae_model = vqvae.utils.load_module(args.vqvae_checkpoint).to(device)


@torch.no_grad
def crop_and_resize_to_square(image, target_size):
    """
    Crops an image tensor to a square shape while maintaining the aspect ratio,
    and then resizes the cropped square image to the target size.
    """
    # Get the dimensions of the image
    height, width = image.shape[-2:]

    # Determine the shorter side
    min_dim = min(height, width)

    # Calculate the crop window
    crop_size = min_dim
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size

    # Crop the image to a square
    image = F.crop(image, top, left, crop_size, crop_size)

    # Resize the cropped square image to the target size
    image = F.resize(image, target_size)

    return image
    
class FlowerDataset(Dataset):
    def __init__(self):
        self.image_size = (64, 64)
        # the dataset is aligned here
        transform = transforms.Compose([
    transforms.RandomResizedCrop(size=(64, 64), scale=(0.8, 1.0), ratio=(1.0, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
        full_dataset = torchvision.datasets.ImageFolder(
            root=os.path.join(data_dir, "102 flower/flowers/train"),
            transform=transform
        )
        #self.tokens = torch.tensor(np.load("work/vqvae/mon_apr_15_11_00_48_2024/flower_image_token.npy")).to(device)
        self.dataset = full_dataset
        m = [
            "pink primrose",
            "hard-leaved pocket orchid",
            "canterbury bells",
            "sweet pea",
            "english marigold",
            "tiger lily",
            "moon orchid",
            "bird of paradise",
            "monkshood",
            "globe thistle",
            "snapdragon",
            "colts foot",
            "king protea",
            "spear thistle",
            "yellow iris",
            "globe-flower",
            "purple coneflower",
            "peruvian lily",
            "balloon flower",
            "giant white arum lily",
            "fire lily",
            "pincushion flower",
            "fritillary",
            "red ginger",
            "grape hyacinth",
            "corn poppy",
            "prince of wales feathers",
            "stemless gentian",
            "artichoke",
            "sweet william",
            "carnation",
            "garden phlox",
            "love in the mist",
            "mexican aster",
            "alpine sea holly",
            "ruby-lipped cattleya",
            "cape flower",
            "great masterwort",
            "siam tulip",
            "lenten rose",
            "barbeton daisy",
            "daffodil",
            "sword lily",
            "poinsettia",
            "bolero deep blue",
            "wallflower",
            "marigold",
            "buttercup",
            "oxeye daisy",
            "common dandelion",
            "petunia",
            "wild pansy",
            "primula",
            "sunflower",
            "pelargonium",
            "bishop of llandaff",
            "gaura",
            "geranium",
            "orange dahlia",
            "pink-yellow dahlia?",
            "cautleya spicata",
            "japanese anemone",
            "black-eyed susan",
            "silverbush",
            "californian poppy",
            "osteospermum",
            "spring crocus",
            "bearded iris",
            "windflower",
            "tree poppy",
            "gazania",
            "azalea",
            "water lily",
            "rose",
            "thorn apple",
            "morning glory",
            "passion flower",
            "lotus",
            "toad lily",
            "anthurium",
            "frangipani",
            "clematis",
            "hibiscus",
            "columbine",
            "desert-rose",
            "tree mallow",
            "magnolia",
            "cyclamen ",
            "watercress",
            "canna lily",
            "hippeastrum ",
            "bee balm",
            "ball moss",
            "foxglove",
            "bougainvillea",
            "camellia",
            "mallow",
            "mexican petunia",
            "bromelia",
            "blanket flower",
            "trumpet creeper",
            "blackberry lily"
        ]
        newm = ["" for i in m]
        idx_to_class = {}
        for m_id in full_dataset.class_to_idx:
            current_id = full_dataset.class_to_idx[m_id]
            idx_to_class[current_id] = m_id
            newm[current_id] = m[int(m_id)-1]
        m = newm
        self.idx_map = {i:f"{idx_to_class[i]}-{m[i]}" for i in range(len(m))}
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, i):
        (path, idx1) = self.dataset.imgs[i]
        image_id = re.findall(r'\d+', path)[-1]
        texts = dd[image_id]
        img, idx1 = self.dataset[i]
        #assert idx1 == idx2
        # text_embeddings = []
        # for t in texts:
        #     if t not in self.text_embedding_cache:
        #         self.text_embedding_cache[t] = self.text_encoder.encode(t).squeeze(0)
        #     text_embeddings.append(self.text_embedding_cache[t])
        return img, None, texts, None, idx1
    @torch.no_grad
    def sample_plot(self, i):
        img, _, texts, _, idx = self[i]
        print(self.idx_map[idx])
        print(img.shape)
        token = vqvae_model.encode_latent(img.unsqueeze(0).to(device))
        img = img.permute((1,2,0)).cpu().detach().numpy()
        img = (1 + img) / 2
        dimg = vqvae_model.decode_latent(token.cuda().unsqueeze(0), self.image_size)
        dimg = dimg.squeeze(0).permute((1,2,0)).cpu().detach().numpy()
        dimg = (dimg + 1) / 2
        f, axes = plt.subplots(1,2)
        axes[0].imshow(img)
        axes[1].imshow(dimg)
        for i in texts:
            print(i)
        for i in axes:
            i.axis('off')
        plt.show()

Working with z of shape (1, 4, 16, 16) = 1024 dimensions.


In [7]:
dataset = FlowerDataset()

In [8]:
import random
@torch.no_grad()
def collate_fn(x):
    text_lists = []
    idxs = []
    raw_imgs = []
    for (raw_img, token, text_list, text_embedding_list, idx) in x:
        raw_imgs.append(raw_img)
        text_lists.append(random.choice(text_list))
        idxs.append(idx)
    return raw_imgs, text_lists, torch.tensor(idx)

dataloader = DataLoader(dataset, batch_size=16, collate_fn=collate_fn)

In [9]:
%load_ext autoreload
%autoreload 3

In [10]:
import mingpt
custom_dataset_info = mingpt.data.CustomDatasetInfo(label_map=None, label_num=0, shape=(64,64,3), vqvae=True, flatten_images=torch.zeros((1,64*64*3), dtype=torch.float).to("cuda"))

In [11]:
if args.checkpoint is None:
    is_training = True
    # if is_training, then we create a new directory automatically
    args.work_dir = os.path.join(args.work_dir, timestamp)
    os.mkdir(args.work_dir)
    for i in ["data", "plot", "runs", "model"]:
        os.mkdir(os.path.join(args.work_dir, i))
    with open(os.path.join(args.work_dir, "config.json"), 'w') as f:
        json.dump(args.__dict__, f, indent=4)
else:
    # otherwise we won't create the working directory
    is_training = False

In [12]:
random.seed(args.seed)
import numpy as np
np.random.seed(args.seed)
import torch
torch.manual_seed(args.seed)

In [13]:
vqvae_model

UNetVQVAE(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down): ModuleList(
      (0): Module(
        (block): ModuleList(
          (0): ResnetBlock(
            (norm1): GroupNorm(32, 32, eps=1e-06, affine=True)
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 32, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          )
        )
        (attn): ModuleList()
        (downsample): Downsample(
          (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2))
        )
      )
      (1): Module(
        (block): ModuleList(
          (0): ResnetBlock(
            (norm1): GroupNorm(32, 32, eps=1e-06, affine=True)
            (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 64

In [14]:
if args.unconditional:
    text_codebook_size = 0
else:
    text_codebook_size = 1

if custom_dataset_info.vqvae:
    assert args.vqvae_checkpoint != None
    if isinstance(vqvae_model, VQVAE):
        image_codebook_size = 0
        for i in vqvae_model.codebooks:
            ci = cast(CodeLayer, i)
            image_codebook_size += ci.n_embed
    else:
        assert isinstance(vqvae_model, UNetVQVAE)
        image_codebook_size = vqvae_model.codebook.n_embed
    token_length = len(vqvae_model.get_latent_offset( custom_dataset_info.shape[:-1]))
else:
    image_codebook_size = 256
    token_length = math.prod(custom_dataset_info.shape)

total_vocab_size = image_codebook_size + 2

In [15]:
config = gen.ExperimentConfig(
    work_dir = args.work_dir,
    checkpoint = args.checkpoint,
    shape = custom_dataset_info.shape,
    image_token_length = token_length,
    # one class token, two special tokens and image tokens
    total_length = token_length + 2 + (not args.unconditional),
    special_token_num = 2,
    bos_id = image_codebook_size,
    eos_id = image_codebook_size + 1,
    row=args.row,
    temperature=args.temperature,
    num_beams=args.num_beams,

    dataset_type = args.dataset_type,
    downscale=args.downscale,

    # tunable hyperparameter for training
    batch_size = args.batch_size,
    learning_rate = args.learning_rate,
    n_epoch = args.n_epoch,
    n_warmup =  args.n_warmup,
    pdropout = args.pdropout,

    log_iter = args.log_iter,
    save_iter = args.save_iter,
    eval_iter = args.eval_iter,

    # hyperparameter for model (default is 3M)
    n_embed = args.n_embed,
    n_head = args.n_head,
    n_layer = args.n_layer,
    use_vqvae = custom_dataset_info.vqvae, # whether use vqvae to compress the image first
    vqvae_checkpoint = args.vqvae_checkpoint,
    total_vocab_size = total_vocab_size,
    image_codebook_size = image_codebook_size,
    text_codebook_size=text_codebook_size,
    unconditional=args.unconditional
)
config

ExperimentConfig(work_dir='work/mingpt/wed_apr_17_21_10_01_2024', checkpoint=None, shape=(64, 64, 3), image_token_length=256, total_length=259, special_token_num=2, bos_id=256, eos_id=257, row=5, temperature=1.0, num_beams=2, dataset_type='vqvae-flower', downscale=1, batch_size=16, learning_rate=0.0006, n_epoch=256, pdropout=0.01, n_warmup=3000, log_iter=300, save_iter=1000, eval_iter=3000, n_embed=256, n_head=8, n_layer=4, use_vqvae=True, vqvae_checkpoint='work/vqvae/mon_apr_15_11_00_48_2024/results/flower-vqvae.pt', total_vocab_size=258, image_codebook_size=256, text_codebook_size=1, unconditional=False)

In [16]:
training_data, eval_data = random_split(dataset, [0.9, 0.1])

In [17]:
def get_text_encoding(x, is_train=True):
    if is_train:
        text_encoder.train()
    else:
        text_encoder.eval()
    s = [torch.tensor([bos_id] +[char_to_ix[char] for char in i] + [eos_id]) for i in x]
    maxlen = max([len(i) for i in s])
    s = [torch.nn.functional.pad(i, (0, maxlen-len(i)), mode='constant', value=pad_id) for i in s]
    _, hidden_state = text_encoder(torch.stack(s), None)
    return hidden_state.permute((1,0,2)).flatten(start_dim=1)

In [86]:
from nanogpt.cond_gpt2 import GPT, GPTConfig
from mingpt.gen import ExperimentConfig, VQVAEType
@torch.no_grad
def generate_images(config : ExperimentConfig,
                    model,
                    vqvae_model : VQVAEType,
                    n_iter,
                    custom_dataset : data.CustomDatasetInfo,
                    row=10,
                    temperature=1.0,
                    num_beams=2):
    bs = row * row
    model.eval()
    model.to(device)
    total_length = config.total_length
    prompt = 'this flower has deep blue blue blue petals and stamen and green leaves.'
    text_embedding = get_text_encoding([prompt], is_train=False).to(device)
    text_embedding = text_embedding.repeat((bs,1))
    start_token = torch.full((bs,1), config.bos_id).to(device)
    inputs = start_token
    result = model.generate(inputs=inputs, max_length=config.image_token_length + 2,text_embedding=text_embedding, min_length=config.image_token_length + 2, top_k=100, temperature=temperature)
    image_height, image_width, image_channel = config.shape
    result = result[:, 1:-1]
    assert result.size(1) == config.image_token_length
    vqvae_model = cast(VQVAE, vqvae_model).to(device)
    vqvae_model.eval()
    off = vqvae_model.get_latent_offset(config.shape[:-1]).to(device)
    result -= off
    outputs = vqvae_model.decode_latent(result, config.shape[:-1]) # (b, c, w, h)
    # outputs is in [-1, 1]
    outputs = ((outputs + 1) / 2) 
    images = outputs.permute((0, 2,3,1)) # (b, w, h, c)
    flatten_images = outputs.flatten(start_dim=1) # (b, c, w, h)
    images = np.maximum(0, np.minimum(1, images.detach().cpu().numpy()))
    
    image_height, image_width, image_channel = config.shape
    # for pixel space image, we clip the value into [0, 255], sometimes the model generates special tokens
    # search image in the database
    flatten_database = custom_dataset.flatten_images
    indices = torch.argmin(torch.cdist(flatten_images, flatten_database), dim = -1)
    match_images = flatten_database[indices].reshape((bs, image_channel, image_height, image_width)).permute((0, 2,3,1)).detach().cpu().numpy()
    ncol = row
    nrow = row
    f, axarr = plt.subplots(nrow,2*ncol, figsize=(ncol*2*2,nrow*2))
    for i in range(nrow):
        for j in range(ncol):
            d = i *ncol+j
            ax = axarr[i][j]
            data = images[d]
            if data.shape[-1] == 3:
                ax.imshow(data, interpolation='none')
            else:
                ax.imshow(data, interpolation='none', cmap="gray")
            ax.axis('off')
    match_labels = custom_dataset.labels
    if match_labels is not None:
        match_labels = match_labels[indices]
    for i in range(nrow):
        for j in range(ncol):
            d = i *ncol+j
            ax = axarr[i][ncol+j]
            data = match_images[d]
            if match_labels is not None:
                iii : int = cast(int, match_labels[d].item())
                label = custom_dataset.label_map[iii]
                ax.title.set_text(label)
            if data.shape[-1] == 3:
                ax.imshow(data, interpolation='none')
            else:
                ax.imshow(data, interpolation='none', cmap="gray")
            ax.axis('off')
    plt.savefig(os.path.join(config.work_dir, f"plot/{n_iter}.png"), dpi=90)
    plt.close(f)
        # Clear the current axes.
    plt.cla() 
    # Clear the current figure.
    plt.clf() 
    # Closes all the figure windows.
    plt.close('all')
generate_images(
    config,
    model,
    vqvae_model,
    "eval",
    custom_dataset_info,
    row=config.row,
    temperature=config.temperature,
    num_beams=config.num_beams)

In [19]:
ts = []
for i in dd:
    ts.extend(dd[i])
raw_text = ""
for i in ts:
    raw_text+=i  
chars = sorted(list(set(raw_text)))
bos_id = len(chars)
chars.append("<BOS>")
pad_id = len(chars)
chars.append("<PAD>")
eos_id = len(chars)
chars.append("<EOS>")
data_size, vocab_size = len(raw_text), len(chars)
print("----------------------------------------")
print("Data has {} characters, {} unique".format(data_size, vocab_size))
print("----------------------------------------")
    
# char to index and index to char maps
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

----------------------------------------
Data has 6079976 characters, 60 unique
----------------------------------------


In [20]:
class RNN(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size, num_layers):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, input_size)
        self.rnn = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.decoder = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, input_seq, hidden_state):
        # input_seq is of shape (b, l)
        embedding = self.embedding(input_seq) # of shape (b, l, e)
        # we use batch first
        output, hidden_state = self.rnn(embedding, hidden_state)
        # output is of shape (b, l, vocab_size)
        output = self.decoder(output)
        # hidden is of shape (num_layer, b, hidden_size)
        return output, hidden_state
hidden_size = 128    # size of hidden state
num_layers = 3       # num of layers in LSTM layer stack
lr = 2e-4            # learning rate
text_encoder = RNN(vocab_size, input_size=vocab_size, hidden_size=hidden_size, num_layers=num_layers)
text_encoder.train()
text_encoder_optimizer = torch.optim.Adam(text_encoder.parameters(), lr=lr)

In [ ]:
# for consistency, we enable bias
model_config = GPT2Config(
    bos_id = config.bos_id,
    eos_id = config.eos_id,
    block_size = config.total_length,
    vocab_size = config.image_codebook_size + 2,
    n_layer = config.n_layer,
    n_head = config.n_head,
    n_embd = config.n_embed,
    dropout = config.pdropout,
    n_text_embd = hidden_size * num_layers,
    bias = True)
model = GPT(model_config)
model

generate_images(config,
                    model,
                    vqvae_model,
                    "eval",
                    custom_dataset_info,
                    row=config.row,
                    temperature=config.temperature,
                    num_beams=config.num_beams)
def train(config : ExperimentConfig,
          model, 
          vqvae_model : VQVAEType,
          optimizer, 
          lr_scheduler, 
          train_dl, 
          eval_dl, 
          custom_dataset : data.CustomDatasetInfo,
          n_iter = 0):
    from torch.utils.tensorboard.writer import SummaryWriter
    writer = SummaryWriter(log_dir=os.path.join(config.work_dir, 'runs'))
    bs = config.batch_size
    num_training_steps = len(train_dl) * config.n_epoch
    progress_bar = tqdm(range(num_training_steps))
    start_token = torch.full((bs,1), config.bos_id).to(device)
    eos_token = torch.full((bs,1), config.eos_id).to(device)
    model.to(device)
    print(config)
    # you can see the config in tensorboard's text section
    writer.add_text('experiment config', str(config))
    writer.add_text('model_size', str(sum([i.numel() for i in model.parameters()]))) 
    log_iter = config.log_iter
    eval_iter = config.eval_iter
    save_iter = config.save_iter
    offset_f = lambda x : x + config.image_codebook_size
    for epoch in range(config.n_epoch):
        for (raw_imgs, texts, class_label) in train_dl:
            batch = vqvae_model.encode_latent(torch.stack(raw_imgs).to(device))
            batch = batch.to(device)
            text_embedding = get_text_encoding(texts, is_train=True).to(device)
            class_label = class_label.to(device)
            model.train()
            text_encoder.train()
            optimizer.zero_grad()
            text_encoder_optimizer.zero_grad()
            batch = torch.cat([start_token, batch, eos_token], dim=-1)
            target = batch.clone()
            assert batch.shape[-1] == config.total_length - 1
            outputs = model(input_ids = batch, labels = target, text_embedding = text_embedding)
            loss = outputs.loss
            loss.backward()
            
            # gradient clip
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(text_encoder.parameters(), 1.0)
            optimizer.step()
            text_encoder_optimizer.step()
            lr_scheduler.step()
            
            progress_bar.update(1)
            
            writer.add_scalar('train_loss', loss.item() ,n_iter)
            writer.add_scalar('lr', lr_scheduler.get_last_lr()[-1] ,n_iter)
            if n_iter % log_iter == 0 or n_iter == num_training_steps - 1:
                generate_images(
                    config,
                    model,
                    vqvae_model,
                    n_iter,
                    custom_dataset,
                    row=config.row,
                    temperature=config.temperature,
                    num_beams=config.num_beams)
            if n_iter % save_iter == 0:
                torch.save(model.state_dict(), os.path.join(config.work_dir, f'model/{n_iter}.pt'))
            # if n_iter % eval_iter == 0 or n_iter == num_training_steps - 1 :
            #     total_loss = 0
            #     for (batch, text_embedding, class_label) in eval_dl:
            #         batch = batch.to(device)
            #         text_embedding = text_embedding.to(device)
            #         class_label = class_label.to(device)
            #         model.eval()
            #         with torch.no_grad():
            #             class_label = offset_f(class_label)
            #             batch = torch.cat([start_token, batch, eos_token], dim=-1)
            #             target = batch.clone()
            #             outputs = model(input_ids = batch, labels = batch, text_embedding = text_embedding)
            #             total_loss += outputs.loss
            #     writer.add_scalar('val_loss', total_loss / len(eval_dl) ,n_iter)
            n_iter += 1
    torch.save(model.state_dict(), os.path.join(config.work_dir, f'model/final.pt'))

optimizer = torch.optim.AdamW(model.parameters(), betas=(0.9,0.95), lr=config.learning_rate, weight_decay=0.1)
num_epochs = config.n_epoch
bs = config.batch_size
train_dl = DataLoader(training_data, batch_size=bs, shuffle=True, drop_last=True, collate_fn=collate_fn, prefetch_factor=5, num_workers=5)
eval_dl =  DataLoader(eval_data, batch_size=bs, shuffle=False, drop_last=True, collate_fn=collate_fn)
device = "cuda"
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = transformers.get_constant_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config.n_warmup
    )
train(config, 
        model,
        vqvae_model, 
        optimizer, 
        lr_scheduler, 
        train_dl, 
        eval_dl, 
        custom_dataset_info,
        n_iter=0)

number of parameters: 3.32M


2024-04-17 21:10:04.285735: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-17 21:10:04.315404: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-17 21:10:04.793973: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


  0%|          | 0/94208 [00:00<?, ?it/s]

ExperimentConfig(work_dir='work/mingpt/wed_apr_17_21_10_01_2024', checkpoint=None, shape=(64, 64, 3), image_token_length=256, total_length=259, special_token_num=2, bos_id=256, eos_id=257, row=5, temperature=1.0, num_beams=2, dataset_type='vqvae-flower', downscale=1, batch_size=16, learning_rate=0.0006, n_epoch=256, pdropout=0.01, n_warmup=3000, log_iter=300, save_iter=1000, eval_iter=3000, n_embed=256, n_head=8, n_layer=4, use_vqvae=True, vqvae_checkpoint='work/vqvae/mon_apr_15_11_00_48_2024/results/flower-vqvae.pt', total_vocab_size=258, image_codebook_size=256, text_codebook_size=1, unconditional=False)


/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


In [79]:
all_texts

['the flower is pink with petals that are soft, smooth, thin and separately arranged around stamens',
 'the flower shown has pink petals with white pistil and green pedicel',
 'a light pink flower with dark pink spots and long pink filaments.',
 'this flower is pink in color, with petals that are spotted.',
 'the petals of the flower are pink in color and have thin filaments that are pink in color.',
 'this flower is pink in color, and has petals that are oval shaped.',
 'this flower has petals of light pink with one petal spotted with darker pink.',
 'this is a light pink flower with dark pink spots on the inside of one petal.',
 'this flower has petals that are pink and has red dots',
 'these flowers have rumpled pink petals with dark pinks pots on some of them.',
 '',
 'the star shaped pink flower has petals that are soft, smooth and has stamens sticking out from the centre',
 'this flower has petals that are pink with purple spots and pink stamen',
 'this flower is pink in color, w

In [ ]:
1